In [ ]:
import networkx as nx
import numpy as np
import pandas as pd

In [ ]:
def has_path(components, u, v):
    if u not in components or v not in components:
        raise ValueError("Node does not exist.")
    return components[u] == components[v]

In [ ]:
version = 1
folder_hidden = "hidden"
output_names = [f"{folder_hidden}/matched_encounters_{version}.txt", f"{folder_hidden}/matched_encounters.txt"]
baseline = "Kostas"
answers_all = {
    "Kostas": pd.read_csv(f"{folder_hidden}/answers_kostas_{version}.csv"),
    "Jon": pd.read_csv(f"{folder_hidden}/answers_jon_{version}.csv"),
}
answers_baseline = answers_all[baseline]
names = answers_all.keys()

In [ ]:
cols_check = np.setdiff1d(answers_all[baseline].columns, ["answer", "time"])

for name, answers in answers_all.items():
    if name == baseline:
        continue
    assert set(answers.columns) == set(answers_baseline.columns)
    assert answers[cols_check].equals(answers_baseline[cols_check])

In [ ]:
components_all = {}
components_inverse_all = {}
for name, answers in answers_all.items():
    identity1 = answers["identity1"].unique()
    identity2 = answers["identity2"].unique()
    identity = np.union1d(identity1, identity2)
    mask = answers["answer"] == "same"
    same = answers.loc[mask, ["identity1", "identity2"]].to_numpy()

    G = nx.Graph()
    G.add_nodes_from(identity.tolist())
    G.add_edges_from(same.tolist())

    components = {}
    components_inverse = {}
    for i, comp in enumerate(nx.connected_components(G)):
        components_inverse[i] = list(comp)
        for node in comp:
            components[node] = i
    components_all[name] = components
    components_inverse_all[name] = components_inverse

components_baseline = components_all[baseline]
components_inverse_baseline = components_inverse_all[baseline]

In [ ]:
for name, answers in answers_all.items():
    groups = list(components_all[name].values())
    n_individuals = len(np.unique(groups))
    n_recaptured_individuals = (pd.Series(groups).value_counts() > 1).sum()
    time = answers["time"].sum() / 60
    # print(f"{name} = {n_individuals} individuals")
    print(f"{name} = {n_recaptured_individuals} individuals with recaptures")
    print(f"{name} = {np.round(time, 2)} minutes")

In [ ]:
for name, answers in answers_all.items():
    mask_different = answers["answer"] == "different"
    different = answers.loc[mask_different, ["identity1", "identity2"]].to_numpy()

    components = components_all[name]
    for u, v in different:
        if has_path(components, u, v):
            print(f"{name}: {u} and {v} conflict")

In [ ]:
already_printed = []
text_to_print = []
for name, answers in answers_all.items():
    if name == baseline:
        continue
    components = components_all[name]
    for u, v in answers[["identity1", "identity2"]].to_numpy():
        connected1 = has_path(components_baseline, u, v)
        connected2 = has_path(components, u, v)
        str1 = "connected" if connected1 else "not connected"
        str2 = "connected" if connected2 else "not connected"
        if connected1 != connected2:
            if {u, v} not in already_printed:
                text = f"{min(u, v)} and {max(u, v)} - {baseline} {str1}. {name} {str2}"
                text_to_print.append(text)
                already_printed.append({u, v})
np.sort(text_to_print).tolist()

In [ ]:
identities = list(G.nodes)
groups_same = set()
groups_diff = set()

for name, answers in answers_all.items():
    if name == baseline:
        continue
    components = components_all[name]
    components_inverse = components_inverse_all[name]
    for identity in identities:
        cluster = components[identity]
        cluster_baseline = components_baseline[identity]
        comp1 = sorted(components_inverse[cluster])
        comp2 = sorted(components_inverse_baseline[cluster_baseline])
        if comp1 == comp2:
            groups_same.add(tuple(sorted(comp1)))
        else:
            groups_diff.add((tuple(sorted(comp1)), tuple(sorted(comp2))))

In [ ]:
parent = {}


def find(x):
    parent.setdefault(x, x)
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]


def union(a, b):
    parent[find(a)] = find(b)


# connect all IDs appearing in the same entry
for group1, group2 in groups_diff:
    ids = list(group1) + list(group2)
    for n in ids[1:]:
        union(ids[0], n)

# group entries by connected component
groups = {}
for entry in groups_diff:
    group1, group2 = entry
    root = find(group1[0])
    groups.setdefault(root, []).append(entry)

for i, group in enumerate(groups.values(), start=1):
    print(f"\nGROUP {i}")
    for entry in group:
        print(entry)

In [ ]:
groups_same1 = list(groups_same)
groups_same2 = [
    (321595638, 333572365, 340438955, 360881828),
    (322750920, 325746516, 326931769, 326931796, 328448933, 330477144, 335332240, 335935552, 335935581, 340435704, 340438919, 340440034, 342033421, 342033424, 345872317, 345874735, 348456347, 351064522, 351064535, 351066969, 360881853),
    (324994660, 328448953, 328755868, 330433249, 330477138, 330477140, 330477174, 337429969, 340438932, 342033439, 342033449),
    (325746501, 328448927, 328448928, 335935550, 342033437, 345874738),
    (325746542, 337429979, 337524320, 340438954),
    (326931767, 326931797, 328448932, 328755826, 330477142, 335935568, 340435714, 340440039, 345872322, 351066973),
    (328755836, 330433239, 330433240, 337429962, 337524316, 340435720, 342033432, 345872311, 348456345, 351066968, 351066978, 360881839),
    (330433230, 342033431),
    (333572353, 335332228, 335935566, 340435706, 340435711, 340440023, 348456349),
    (340435712, 345874739, 348456353, 351066971),
    (351066982, 360875362),
    (328448946, 328448947, 328755818, 328755846, 330430787, 330477160, 330477162),
]

In [ ]:
groups = groups_same1 + groups_same2
groups = list({tuple(sorted(t)) for t in groups})
for output_name in output_names:
    with open(output_name, "w") as f:
        f.write(str(groups))